# PimaGuard — Analise Exploratoria e Modelagem
**Disciplina:** Inteligencia Artificial Aplicada a Saude  
**Dataset:** Pima Indians Diabetes Database (UCI / Kaggle)  
**Tema:** Diabetes Tipo 2  
**Universidade:** Sao Judas Tadeu — 1 Semestre 2026

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'neg': '#3B82F6', 'pos': '#EF4444', 'teal': '#0D9488'}
print('Imports OK')

## 1. Carregamento dos Dados

In [ ]:
from src.data_loader import load_data, FEATURE_NAMES, FEATURE_LABELS, ZERO_INVALID_COLS

df_raw = load_data()
print(f'Shape: {df_raw.shape}')
print(f'Colunas: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
print('Distribuicao da variavel alvo:')
vc = df_raw['Outcome'].value_counts()
print(f'  Nao-diabetico (0): {vc[0]} ({vc[0]/len(df_raw)*100:.1f}%)')
print(f'  Diabetico     (1): {vc[1]} ({vc[1]/len(df_raw)*100:.1f}%)')
print()
print('Estatisticas descritivas:')
df_raw.describe().round(2)

## 2. Analise de Dados Faltantes e Zeros Invalidos

In [ ]:
zeros = pd.DataFrame({
    'Variavel': ZERO_INVALID_COLS,
    'Zeros': [(df_raw[c] == 0).sum() for c in ZERO_INVALID_COLS],
    'Percentual (%)': [(df_raw[c] == 0).mean() * 100 for c in ZERO_INVALID_COLS],
})
zeros['Percentual (%)'] = zeros['Percentual (%)'].round(1)
print('Zeros invalidos por variavel:')
display(zeros)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(zeros['Variavel'], zeros['Percentual (%)'], color=PALETTE['pos'], alpha=0.8)
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_xlabel('Percentual de zeros (%)')
ax.set_title('Zeros invalidos que serao imputados')
plt.tight_layout()
plt.show()

## 3. Pre-processamento

In [ ]:
from src.preprocessing import handle_zeros, impute_by_class, preprocess, split_data

df = handle_zeros(df_raw)
df = impute_by_class(df)

print('Nulos apos imputacao:', df[FEATURE_NAMES].isnull().sum().sum())
print()
print('Comparacao de medianas antes x depois (Insulina):')
print(f'  Antes: {df_raw["Insulin"].median():.1f}  |  Depois: {df["Insulin"].median():.1f}')

X, y, scaler = preprocess(df_raw)
X_train, X_test, y_train, y_test = split_data(X, y)
print(f'\nTreino: {X_train.shape[0]}  |  Teste: {X_test.shape[0]}')

## 4. Analise Exploratoria (EDA)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_NAMES):
    ax = axes[i]
    for cls, label, color in [(0, 'Nao-diabetico', PALETTE['neg']), (1, 'Diabetico', PALETTE['pos'])]:
        vals = df[df['Outcome'] == cls][feat]
        ax.hist(vals, bins=25, alpha=0.65, color=color, label=label, density=True)
    ax.set_title(FEATURE_LABELS[feat], fontweight='bold', fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=7)

plt.suptitle('Distribuicao das Variaveis por Classe', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[FEATURE_NAMES + ['Outcome']].corr()
labels_pt = [FEATURE_LABELS.get(f, f) for f in FEATURE_NAMES] + ['Diabetes']

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax, square=True,
    xticklabels=labels_pt, yticklabels=labels_pt,
    linewidths=0.5, cbar_kws={'shrink': 0.8},
)
ax.set_title('Matriz de Correlacao', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()
print('Correlacao com Diabetes:')
print(corr['Outcome'][FEATURE_NAMES].sort_values(ascending=False).to_string())

## 5. Treinamento dos Modelos

In [ ]:
from src.train import train_sklearn, train_keras, load_models, MODELS_DIR
import joblib

os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.pkl'))

print('Treinando modelos sklearn...')
models = train_sklearn(X_train, y_train)
print(f'{len(models)} modelos treinados.')

print('\nTreinando Rede Neural Keras...')
keras_model = train_keras(X_train, y_train)
if keras_model:
    models['Rede Neural Keras'] = keras_model
    print('Keras treinado.')

## 6. Avaliacao dos Modelos

In [ ]:
from src.evaluate import evaluate_all, cross_validate_sklearn

results = evaluate_all(models, X_test, y_test)

print(f'{'Modelo':<25} {'Acuracia':>9} {'Precisao':>9} {'Recall':>9} {'F1':>9} {'AUC':>9}')
print('-' * 66)
for name, data in results.items():
    m = data['metrics']
    print(f"{name:<25} {m['Acuracia']:>9.4f} {m['Precisao']:>9.4f} {m['Recall']:>9.4f} {m['F1-Score']:>9.4f} {m['ROC-AUC']:>9.4f}")

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = [PALETTE['teal'], PALETTE['neg'], '#F59E0B', PALETTE['pos'], '#8B5CF6']
ax_roc = axes[0]
ax_roc.plot([0,1], [0,1], 'k--', lw=1, label='Referencia aleatoria')
for i, (name, data) in enumerate(results.items()):
    fpr, tpr = data['roc']
    auc = data['metrics']['ROC-AUC']
    ax_roc.plot(fpr, tpr, lw=2, color=palette[i % len(palette)], label=f'{name} (AUC={auc:.3f})')
ax_roc.set_xlabel('Taxa de Falso Positivo')
ax_roc.set_ylabel('Taxa de Verdadeiro Positivo')
ax_roc.set_title('Curvas ROC', fontweight='bold')
ax_roc.legend(fontsize=8)

ax_bar = axes[1]
metric_keys = ['Acuracia', 'Precisao', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metric_keys))
bw = 0.8 / len(results)
for j, (name, data) in enumerate(results.items()):
    vals = [data['metrics'][k] for k in metric_keys]
    ax_bar.bar(x + (j - len(results)/2 + 0.5)*bw, vals, width=bw*0.9,
               label=name, color=palette[j % len(palette)], alpha=0.85)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(metric_keys, rotation=15)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_title('Comparacao de Metricas', fontweight='bold')
ax_bar.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from src.evaluate import predict_class

n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, data) in zip(axes, results.items()):
    cm = data['confusion_matrix']
    disp = ConfusionMatrixDisplay(cm, display_labels=['Nao-diab.', 'Diabetico'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=9, fontweight='bold')

plt.suptitle('Matrizes de Confusao', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Validacao Cruzada (5-Fold)

In [ ]:
cv_results = {}
for name, model in models.items():
    if 'Keras' not in name:
        cv = cross_validate_sklearn(model, X, y)
        cv_results[name] = cv
        print(f'{name:<25}  AUC: {cv["auc_mean"]:.4f} +/- {cv["auc_std"]:.4f}  |  Acc: {cv["acc_mean"]:.4f} +/- {cv["acc_std"]:.4f}')

## 8. Explicabilidade com SHAP

In [ ]:
import shap

rf_model = models.get('Random Forest')
if rf_model:
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test)
    shap_vals_pos = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    plt.figure(figsize=(9, 5))
    shap.summary_plot(
        shap_vals_pos, X_test,
        feature_names=[FEATURE_LABELS[f] for f in FEATURE_NAMES],
        plot_type='bar', show=False, color=PALETTE['teal']
    )
    plt.title('Importancia Global das Variaveis (SHAP)', fontweight='bold')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_vals_pos, X_test,
        feature_names=[FEATURE_LABELS[f] for f in FEATURE_NAMES],
        show=False
    )
    plt.title('Distribuicao dos Valores SHAP por Variavel', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Conclusoes

**Principais achados:**
- **Glicose** e a variavel com maior correlacao com diabetes (r=0.47) e maior importancia nos modelos baseados em arvore.
- **IMC** e **Idade** tambem sao preditores relevantes, consistentes com a literatura clinica.
- O **Random Forest** e **Gradient Boosting** apresentaram desempenho superior na maioria das metricas.
- Validacao cruzada 5-Fold confirma a robustez dos resultados.

**Limitacoes:**
- Dataset restrito a mulheres Pima — baixa generalizabilidade.
- Presenca significativa de zeros invalidos que exigiram imputacao.
- Desequilibrio de classes (65/35) pode subestimar o recall em diabeticos.

**Proximos passos:**
- Incluir dados do DATASUS para validacao em populacao brasileira.
- Explorar SMOTE para mitigar desequilibrio de classes.
- Calibrar probabilidades para uso clinico real.
- Implementar monitoramento de drift do modelo em producao.